# 04 - Extract Open-Meteo Weather Data and Build Daily Weather Fact Table

## Overview

This notebook retrieves weather data for the geographic areas defined in the Renewables-Climate Mart and transforms the observations into a daily weather fact table.

Weather conditions play an important role in renewable electricity generation, particularly for wind and solar assets. To provide contextual information for subsequent analyses, hourly weather observations are obtained from the Open-Meteo Archive API for all geographic areas contained in the area dimension.

The extracted weather data includes temperature, wind speed, surface pressure, and shortwave radiation. These observations are subsequently transformed into daily indicators and enriched with derived metrics such as air density and wind power density.

The resulting weather fact table can be linked to the generation data through shared geographic and temporal dimensions, enabling analyses of weather-related generation patterns within the Renewables-Climate Mart.

## Data Source

This notebook uses historical weather data provided by the Open-Meteo Archive API:

https://open-meteo.com/

The geographic coordinates used for the API requests are obtained from the area dimension created in Notebook 02:

* `data/processed/dim_area_h.csv`

## Output

The notebook exports the daily weather fact table to:

* `data/processed/fact_weather_daily.csv`


In [1]:
import pandas as pd

import openmeteo_requests
import requests_cache
from retry_requests import retry

import os
import time
import math

In [2]:
DATA_DIR = "../data"

RAW_OPENMETEO_DIR = f"{DATA_DIR}/raw/openmeteo"
CACHE_DIR = f"{RAW_OPENMETEO_DIR}/cache"
PROCESSED_DIR = f"{DATA_DIR}/processed"

In [3]:
dim_area_h = pd.read_csv(f"{PROCESSED_DIR}/dim_area_h.csv")

## 1. Extract Hourly Weather Data from Open-Meteo

This section retrieves hourly weather observations from the Open-Meteo Archive API for all geographic areas defined in the area dimension.

If a previously extracted raw weather file already exists, it is loaded directly from the raw data folder to avoid unnecessary API requests. Otherwise, the notebook requests the data from Open-Meteo in multiple chunks to reduce the size of individual API calls. The extracted variables include temperature, wind speed at 100 meters, shortwave radiation, and surface pressure.

Since the Open-Meteo API returns timestamps in UTC, weather data is requested until 1 January 2026 to ensure complete coverage of the final local hours of 2025 after conversion to the Europe/Berlin timezone. Observations outside the 2024–2025 analysis period are subsequently removed.

The resulting dataset is stored as a raw weather extract and serves as the basis for the daily weather fact table created in the next section.

In [4]:
# Taken and modified from open-meteo.com/en/docs/historical-weather-api

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession(CACHE_DIR, expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=2)
openmeteo = openmeteo_requests.Client(session=retry_session)

def fetch_all_weather(dim_area_h):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    # Request all coordinates in the provided chunk at once.
    params = {
        "latitude": dim_area_h['latitude'].tolist(),
        "longitude": dim_area_h['longitude'].tolist(),
        "start_date": "2024-01-01",
        "end_date": "2026-01-01",
        "hourly": ["wind_speed_100m", "shortwave_radiation", "surface_pressure", "temperature_2m"],
        "timezone": "Europe/Berlin",
    }
    
    # Returns a list of responses (one per coordinate)
    responses = openmeteo.weather_api(url, params=params)
    
    all_data = []
    
    # Process the results by matching index with area_id
    for i, response in enumerate(responses):
        area_id = dim_area_h.iloc[i]['area_id']
        hourly = response.Hourly()
        
        df = pd.DataFrame({
            "date": pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="left"
            ).tz_convert("Europe/Berlin"),
            "wind_speed_100m": hourly.Variables(0).ValuesAsNumpy(),
            "shortwave_radiation": hourly.Variables(1).ValuesAsNumpy(),
            "surface_pressure": hourly.Variables(2).ValuesAsNumpy(),
            "temperature_2m": hourly.Variables(3).ValuesAsNumpy(),
            "area_id": area_id
        })
        all_data.append(df)
    
    return pd.concat(all_data, ignore_index=True)

In [5]:
raw_weather_path = f"{RAW_OPENMETEO_DIR}/openMeteo_windSpeed_shortwaveRadiation_pressure_temperature_hourly.csv"

if os.path.exists(raw_weather_path):
    print("Loading existing Open-Meteo extract from raw data folder.")
    fact_weather = pd.read_csv(raw_weather_path)
    fact_weather["date"] = pd.to_datetime(fact_weather["date"], utc=True).dt.tz_convert("Europe/Berlin")
else:
    # Split 241 areas into roughly 10 chunks (~25 areas per chunk) in order to reduce the payload size per request
    num_chunks = 10
    chunk_size = math.ceil(len(dim_area_h) / num_chunks)

    df_chunks = [
        dim_area_h.iloc[i:i + chunk_size].copy()
        for i in range(0, len(dim_area_h), chunk_size)
    ]

    fact_weather = pd.DataFrame()

    print(f"Starting weather fetch for {len(dim_area_h)} areas across {num_chunks} chunks...")

    for i, chunk in enumerate(df_chunks):
        print(f"Fetching chunk {i+1}/{num_chunks} ({len(chunk)} areas)...")

        try:
            hourly_df = fetch_all_weather(chunk)
            fact_weather = pd.concat([fact_weather, hourly_df], ignore_index=True)

            # Persist progress after every successful chunk
            fact_weather.to_csv(raw_weather_path, index=False)

            # Pause between requests to reduce the likelihood of API rate limiting
            if i < num_chunks - 1:
                print("Request successful. Sleeping before next request...")
                time.sleep(10)

        except Exception as e:
            print(f"Error on chunk {i+1}: {e}")
            print("Partial results saved. Delete the raw extract before restarting the full extraction.")
            break

Loading existing Open-Meteo extract from raw data folder.


In [6]:
fact_weather_filtered = fact_weather[(fact_weather['date'].dt.year >= 2024)&(fact_weather['date'].dt.year < 2026)].copy()
fact_weather_filtered

,date,wind_speed_100m,shortwave_radiation,surface_pressure,temperature_2m,area_id
1,2024-01-01 00:00:00+01:00,35.632725,0.0,980.53650,5.95,1
2,2024-01-01 01:00:00+01:00,34.162083,0.0,980.45030,6.05,1
3,2024-01-01 02:00:00+01:00,36.945190,0.0,981.18130,5.65,1
4,2024-01-01 03:00:00+01:00,39.042990,0.0,981.37510,5.65,1
5,2024-01-01 04:00:00+01:00,40.237950,0.0,981.49994,5.90,1
...,...,...,...,...,...,...
4233860,2025-12-31 19:00:00+01:00,34.100380,0.0,978.98535,0.45,241
4233861,2025-12-31 20:00:00+01:00,35.756172,0.0,978.78600,0.40,241
4233862,2025-12-31 21:00:00+01:00,36.473880,0.0,978.82040,0.65,241
4233863,2025-12-31 22:00:00+01:00,35.481922,0.0,978.73770,0.75,241


## 2. Build Daily Weather Fact Table

This section transforms the hourly weather observations into daily weather indicators.

Air density is calculated from surface pressure and temperature, and hourly wind power density is derived from air density and wind speed. The hourly observations are then aggregated to daily values by area, including average temperature, average wind power density, and total solar irradiation.

The resulting daily weather fact table is validated for missing values and date coverage before being exported.

In [7]:
# Calculate air density and wind power density from hourly weather observations.
R = 287.058

fact_weather_filtered['rho'] = (fact_weather_filtered['surface_pressure'] * 100) / (R * (fact_weather_filtered['temperature_2m'] + 273.15))

fact_weather_filtered['wind_speed_ms'] = fact_weather_filtered['wind_speed_100m'] / 3.6
fact_weather_filtered['hourly_wpd'] = 0.5 * fact_weather_filtered['rho'] * (fact_weather_filtered['wind_speed_ms']**3)


# Aggregate hourly observations to daily area-level weather indicators.
date_key = fact_weather_filtered['date'].dt.strftime('%Y%m%d').astype(int)

fact_weather_daily = fact_weather_filtered.groupby(['area_id', date_key]).agg(
    avg_temperature_C=('temperature_2m', 'mean'),
    avg_wind_power_density_W_per_m2=('hourly_wpd', 'mean'),
    total_solar_irradiation_kWh_per_m2=('shortwave_radiation', lambda x: x.sum() / 1000)
).reset_index()

fact_weather_daily.rename(columns={'date': 'date_id'}, inplace=True)

In [8]:
fact_weather_daily.isna().sum()

area_id                               0
date_id                               0
avg_temperature_C                     0
avg_wind_power_density_W_per_m2       0
total_solar_irradiation_kWh_per_m2    0
dtype: int64

In [9]:
expected_days = 731

coverage = fact_weather_daily.groupby("area_id")["date_id"].nunique()

assert coverage.min() == expected_days and coverage.max() == expected_days, \
    "Unexpected weather date coverage."

In [10]:
fact_weather_daily.to_csv(f"{PROCESSED_DIR}/fact_weather_daily.csv", index=False)
print(f"Exported {len(fact_weather_daily):,} daily weather records.")

Exported 176,171 daily weather records.
